In [ ]:
# 03_baseline.ipynb -- Study 1 LightGBM baseline
# !pip install lightgbm -q

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error

from features import build_feature_table, make_next_day_target, TARGET

df = pd.read_csv("data/study1_daily.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
feat_df = build_feature_table(df)
feat_df = make_next_day_target(feat_df)  # TARGET -> tomorrow's actual demand; TARGET_today preserved

feature_cols = [c for c in feat_df.columns if
                c.startswith(f"{TARGET}_lag") or
                c.startswith(f"{TARGET}_roll") or
                c in ("dow", "month", "year", "is_weekend", "day_of_year")]

feat_df = feat_df.dropna(subset=[f"{TARGET}_lag365", TARGET]).reset_index(drop=True)
# --- Time-aware split ---
train = feat_df[feat_df["year"] <= 2022]
val = feat_df[feat_df["year"] == 2023]
test = feat_df[feat_df["year"] >= 2024]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)

# NOTE: anchor is TARGET_today (today's actual demand), NOT TARGET_lag1.
# lag1 was computed relative to the pre-shift target, so after make_next_day_target()
# it's 2 days behind the new target, not 1 -- see features.py's make_next_day_target()
# docstring for the full story. Using TARGET_today here is what makes both the
# delta-target and the naive-persistence baseline below correctly anchored.
X_train, y_train = train[feature_cols], train[TARGET] - train[f"{TARGET}_today"]
X_val, y_val = val[feature_cols], val[TARGET] - val[f"{TARGET}_today"]
X_test, y_test = test[feature_cols], test[TARGET]

model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

preds_test = model.predict(X_test) + test[f"{TARGET}_today"].values

# --- Naive persistence baseline (today's actual value = tomorrow's prediction) ---
naive_preds = test[f"{TARGET}_today"]

def report(name, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    print(f"{name}: MAPE={mean_absolute_percentage_error(y_true, y_pred):.4f} "
          f"RMSE={rmse:.1f} "
          f"MAE={mean_absolute_error(y_true, y_pred):.1f}")

report("LightGBM", y_test, preds_test)
report("Naive persistence", y_test, naive_preds)

# --- Feature importance ---
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance.head(20))

plt.figure(figsize=(8, 6))
top_imp = importance.head(15)
norm = mpl.colors.Normalize(vmin=top_imp.min(), vmax=top_imp.max())
colors = mpl.colormaps["PuBu"](norm(top_imp.values))
plt.barh(top_imp.index[::-1], top_imp.values[::-1], color=colors[::-1])
plt.xlabel("Split count")
plt.title("Feature importance -- next-day demand forecast")
plt.tight_layout()
plt.show()

# --- Forecast residual (needed by Phase 4) ---
test = test.copy()
test["forecast"] = preds_test
test["residual"] = test[TARGET] - test["forecast"]
test[["date", TARGET, "forecast", "residual"]].to_csv("data/study1_forecast_residual.csv", index=False)
